In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2" 

import sys
sys.path.append("/home/cyf/wbi/Virginia/code")

import json
import torch
from importlib import reload

from CoarseFlow.training import train, losses
from CoarseFlow.datasets import synthetic_dataset

reload(train)
reload(losses)
reload(synthetic_dataset)

from CoarseFlow.training.train import train_coarse_matching_model
from CoarseFlow.datasets.synthetic_dataset import (
    build_sameShape_loader,
    summarize_manifest,
)

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("current device:", torch.cuda.current_device())
    print("device name:", torch.cuda.get_device_name(0))

torch: 2.6.0+cu124
cuda available: True
device count: 1
current device: 0
device name: NVIDIA A100-SXM4-80GB


读取manifest_summary.json
---

In [2]:
# train.ipynb - Cell 2

MANIFEST_SUMMARY_PATH = "cached_datasets/coarseflow_v6/manifest_summary.json"

with open(MANIFEST_SUMMARY_PATH, "r") as f:
    manifest_summary = json.load(f)

for k, v in manifest_summary.items():
    print(k, "->", v)

for name, path in manifest_summary.items():
    print("\n" + "=" * 100)
    print(name)
    summarize_manifest(path)

stage0_train -> cached_datasets/coarseflow_v6/stage0_sanity_train/stage0_train_manifest.json
stage0_val -> cached_datasets/coarseflow_v6/stage0_sanity_val/stage0_val_manifest.json
stage1_train -> cached_datasets/coarseflow_v6/stage1_K5_varSpacing_train/stage1_train_manifest.json
stage1_val -> cached_datasets/coarseflow_v6/stage1_K5_varSpacing_val/stage1_val_manifest.json
stage2_train -> cached_datasets/coarseflow_v6/stage2_varK_varSpacing_train/stage2_train_manifest.json
stage2_val -> cached_datasets/coarseflow_v6/stage2_varK_varSpacing_val/stage2_val_manifest.json
stage3_train -> cached_datasets/coarseflow_v6/stage3_hard_train/stage3_train_manifest.json
stage3_val -> cached_datasets/coarseflow_v6/stage3_hard_val/stage3_val_manifest.json
final_test -> cached_datasets/coarseflow_v6/final_test/final_test_manifest.json

stage0_train
Manifest: cached_datasets/coarseflow_v6/stage0_sanity_train/stage0_train_manifest.json
split   : stage0_train
spacing : per_volume_ref_spacing
parts   : 1
sam

定义模型
---

In [ ]:
# train.ipynb - Cell 4
model_config = dict(
    dim=96,
    radius=(4, 5, 5),
    temperature=0.05,
    use_learned_matching=True,
    matcher_mode="hybrid",

    control_stride=16,
    encoder_stride=4,
    num_refine_iters=1,
    query_chunk_size=128,

    moving_num_convnext_blocks=3,
    moving_window_attn_layers=2,

    # reference attention
    ref_use_attention=True,
    ref_attn_layers=2,
    ref_attn_num_heads=4,
    ref_attn_window_size=(3, 8, 8),
    ref_attn_mlp_ratio=2.0,
    ref_base_channels=(32, 64, 96),
    ref_num_blocks=(2, 4, 4),
    ref_refine_blocks=3,
    use_coord_embed=True,

    use_offset_encoding=True,
    use_offset_bias=True,
    use_local_cross_attn=True,
    local_attn_temperature=0.20,

    use_spacing_embed=True,
    moving_window_attn_layers=2,

    ref_use_attention=True,
    ref_attn_layers=2,
    ref_attn_num_heads=4,
    ref_attn_window_size=(3, 8, 8),
    ref_attn_mlp_ratio=2.0,

    # ----------------------------------------------------
    # key change for stage 3
    # ----------------------------------------------------
    num_refine_iters=3,
)
model_config_attn = dict(model_config)

model_config_attn.update(
    dict(
        # moving attention
        moving_window_attn_layers=2,

        # reference attention
        ref_use_attention=True,
        ref_attn_layers=2,
        ref_attn_num_heads=4,
        ref_attn_window_size=(3, 8, 8),
        ref_attn_mlp_ratio=2.0,
    )

)
model_config

{'dim': 96,
 'radius': (4, 5, 5),
 'temperature': 0.05,
 'use_learned_matching': True,
 'matcher_mode': 'hybrid',
 'control_stride': 16,
 'encoder_stride': 4,
 'num_refine_iters': 1,
 'query_chunk_size': 128,
 'moving_num_convnext_blocks': 3,
 'moving_window_attn_layers': 2,
 'ref_use_attention': True,
 'ref_attn_layers': 2,
 'ref_attn_num_heads': 4,
 'ref_attn_window_size': (3, 8, 8),
 'ref_attn_mlp_ratio': 2.0,
 'ref_base_channels': (32, 64, 96),
 'ref_num_blocks': (2, 4, 4),
 'ref_refine_blocks': 3,
 'use_coord_embed': True,
 'use_offset_encoding': True,
 'use_offset_bias': True,
 'use_local_cross_attn': True,
 'local_attn_temperature': 0.2,
 'use_spacing_embed': True}

Stage 0 sanity training
---

In [4]:
# Stage 0: sanity training

train_loader_stage0, _, _ = build_sameShape_loader(
    manifest_summary["stage0_train"],
    batch_size=5,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
    drop_last=False,
)

val_loader_stage0, _, _ = build_sameShape_loader(
    manifest_summary["stage0_val"],
    batch_size=1,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
    drop_last=False,
)

model_stage0 = train_coarse_matching_model(
    train_dataset=None,
    val_dataset=None,
    train_loader=train_loader_stage0,
    val_loader=val_loader_stage0,

    save_dir="./checkpoints/coarseflow_v6_stage0_sanity",

    num_epochs=20,
    lr=5e-5,
    batch_size=5,
    num_workers=0,
    use_amp=True,

    **model_config,

    loss_mode="match",
    lambda_match=1.0,
    lambda_match_kl=1.0,
    lambda_match_ce=0.3,
    lambda_coord=0.1,
    lambda_disp=0.0,
    lambda_smooth=0.005,
    lambda_z_spacing=0.005,

    resume_path=None,
    resume_optimizer=False,
    strict_load=False,

    log_filename="train.log",
    log_mode="w",
)

[SameShapeBatchSampler] groups:
  key=(K,D,H,W)=(5, 20, 256, 256), n=11
  key=(K,D,H,W)=(5, 30, 256, 256), n=9
[SameShapeBatchSampler] groups:
  key=(K,D,H,W)=(5, 20, 256, 256), n=7
  key=(K,D,H,W)=(5, 30, 256, 256), n=3
2026-05-07 18:18:06 | [Logger] log file: ./checkpoints/coarseflow_v6_stage0_sanity/train.log


/home/cyf/wbi/Virginia/code/CoarseFlow/training/train.py:568: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=use_amp)
/home/cyf/wbi/Virginia/code/CoarseFlow/training/train.py:155: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):


2026-05-07 18:18:08 | [Epoch 001] step 0000/5 | loss=6.5629 | match=5.8684 | kl=3.7027 | ce=7.2189 | coord=6.6259 | disp=6.6259 | smooth=4.3463 | z_spacing=2.0488 | top1=0.002 | top5=0.009 | pmax=0.014 | ent=6.340 | inside_valid=1.000 | cand_valid=0.820 | inside_all=0.864 | valid_inside_all=0.595 | time=2.1s | valid=0.595 | 
2026-05-07 18:18:13 | [Epoch 001] train_loss=6.4154


/home/cyf/wbi/Virginia/code/CoarseFlow/training/train.py:317: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):


2026-05-07 18:18:13 | [Val Epoch 001] loss=6.1079 | match=5.4935 | kl=3.4059 | ce=6.9584 | coord=5.9632 | disp=5.9632 | smooth=2.2494 | z_spacing=1.3738 | top1=0.001 | top5=0.008 | pmax=0.006 | ent=6.645 | inside_valid=0.997 | cand_valid=0.830 | inside_all=0.881 | valid_inside_all=0.628 | valid=0.630 | pred_mag=2.631 | gt_mag=3.064
2026-05-07 18:18:15 | [Epoch 002] step 0000/5 | loss=5.9128 | match=5.3835 | kl=3.3283 | ce=6.8507 | coord=5.0854 | disp=5.0854 | smooth=2.3678 | z_spacing=1.7757 | top1=0.003 | top5=0.010 | pmax=0.007 | ent=6.612 | inside_valid=1.000 | cand_valid=0.820 | inside_all=0.879 | valid_inside_all=0.612 | time=1.2s | valid=0.612 | 
2026-05-07 18:18:19 | [Epoch 002] train_loss=5.8442
2026-05-07 18:18:20 | [Val Epoch 002] loss=5.9152 | match=5.3314 | kl=3.2843 | ce=6.8238 | coord=5.6833 | disp=5.6833 | smooth=1.8130 | z_spacing=1.2776 | top1=0.002 | top5=0.011 | pmax=0.005 | ent=6.706 | inside_valid=0.997 | cand_valid=0.830 | inside_all=0.881 | valid_inside_all=0.628

Stage 1，K=5 + variable spacing
---

In [5]:
# Stage 1: K=5 + variable spacing

train_loader_stage1, _, _ = build_sameShape_loader(
    manifest_summary["stage1_train"],
    batch_size=5,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
    drop_last=False,
)

val_loader_stage1, _, _ = build_sameShape_loader(
    manifest_summary["stage1_val"],
    batch_size=1,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
    drop_last=False,
)

model_stage1 = train_coarse_matching_model(
    train_dataset=None,
    val_dataset=None,
    train_loader=train_loader_stage1,
    val_loader=val_loader_stage1,

    save_dir="./checkpoints/coarseflow_v6_stage1_K5_varSpacing",

    num_epochs=150,
    lr=5e-5,
    batch_size=5,
    num_workers=0,
    use_amp=True,

    **model_config,

    loss_mode="match",
    lambda_match=1.0,
    lambda_match_kl=1.0,
    lambda_match_ce=0.3,
    lambda_coord=0.1,
    lambda_disp=0.0,
    lambda_smooth=0.005,
    lambda_z_spacing=0.005,

    resume_path=None,
    resume_optimizer=False,
    strict_load=False,

    log_filename="train.log",
    log_mode="w",
)

[SameShapeBatchSampler] groups:
  key=(K,D,H,W)=(5, 20, 256, 256), n=339
  key=(K,D,H,W)=(5, 30, 256, 256), n=111
[SameShapeBatchSampler] groups:
  key=(K,D,H,W)=(5, 20, 256, 256), n=71
  key=(K,D,H,W)=(5, 30, 256, 256), n=19
2026-05-07 18:28:21 | [Logger] log file: ./checkpoints/coarseflow_v6_stage1_K5_varSpacing/train.log
2026-05-07 18:28:22 | [Epoch 001] step 0000/91 | loss=6.7266 | match=6.1455 | kl=3.9109 | ce=7.4485 | coord=5.5860 | disp=5.5860 | smooth=3.6323 | z_spacing=0.8756 | top1=0.001 | top5=0.004 | pmax=0.022 | ent=6.269 | inside_valid=1.000 | cand_valid=0.852 | inside_all=0.922 | valid_inside_all=0.662 | time=1.6s | valid=0.662 | 
2026-05-07 18:28:35 | [Epoch 001] step 0010/91 | loss=5.8218 | match=5.3090 | kl=3.2751 | ce=6.7798 | coord=5.0021 | disp=5.0021 | smooth=1.8368 | z_spacing=0.6790 | top1=0.003 | top5=0.014 | pmax=0.004 | ent=6.767 | inside_valid=1.000 | cand_valid=0.856 | inside_all=0.913 | valid_inside_all=0.647 | time=14.8s | valid=0.647 | 
2026-05-07 18:28:

Stage 2，variable K + variable spacing
---

In [ ]:
# Stage 2: variable K + variable spacing

train_loader_stage2, _, _ = build_sameShape_loader(
    manifest_summary["stage2_train"],
    batch_size=4,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
    drop_last=False,
)

val_loader_stage2, _, _ = build_sameShape_loader(
    manifest_summary["stage2_val"],
    batch_size=1,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
    drop_last=False,
)

model_stage2 = train_coarse_matching_model(
    train_dataset=None,
    val_dataset=None,
    train_loader=train_loader_stage2,
    val_loader=val_loader_stage2,

    save_dir="./checkpoints/coarseflow_v6_stage2_varK_varSpacing",

    num_epochs=300,
    lr=3e-5,
    batch_size=4,
    num_workers=0,
    use_amp=True,

    **model_config,

    loss_mode="match",
    lambda_match=1.0,
    lambda_match_kl=1.0,
    lambda_match_ce=0.3,
    lambda_coord=0.1,
    lambda_disp=0.0,
    lambda_smooth=0.005,
    lambda_z_spacing=0.005,

    resume_path="./checkpoints/coarseflow_v6_stage1_K5_varSpacing/best.pth",
    resume_optimizer=False,
    strict_load=True,

    log_filename="train.log",
    log_mode="w",
)

####中断了重新训练
# model_stage2 = train_coarse_matching_model(
#     train_dataset=None,
#     val_dataset=None,
#     train_loader=train_loader_stage2,
#     val_loader=val_loader_stage2,

#     save_dir="./checkpoints/coarseflow_v6_stage2_varK_varSpacing",

#     num_epochs=100,
#     lr=3e-5,
#     batch_size=4,
#     num_workers=0,
#     use_amp=True,

#     **model_config,

#     loss_mode="match",
#     lambda_match=1.0,
#     lambda_match_kl=1.0,
#     lambda_match_ce=0.3,
#     lambda_coord=0.1,
#     lambda_disp=0.0,
#     lambda_smooth=0.005,
#     lambda_z_spacing=0.005,

#     resume_path="./checkpoints/coarseflow_v6_stage2_varK_varSpacing/latest.pth",
#     resume_optimizer=False,
#     strict_load=True,

#     log_filename="train.log",
#     log_mode="w",
# )

[SameShapeBatchSampler] groups:
  key=(K,D,H,W)=(3, 20, 256, 256), n=230
  key=(K,D,H,W)=(3, 30, 256, 256), n=70
  key=(K,D,H,W)=(4, 20, 256, 256), n=235
  key=(K,D,H,W)=(4, 30, 256, 256), n=65
  key=(K,D,H,W)=(5, 20, 256, 256), n=217
  key=(K,D,H,W)=(5, 30, 256, 256), n=83
  key=(K,D,H,W)=(6, 20, 256, 256), n=223
  key=(K,D,H,W)=(6, 30, 256, 256), n=77
  key=(K,D,H,W)=(7, 20, 256, 256), n=224
  key=(K,D,H,W)=(7, 30, 256, 256), n=76
[SameShapeBatchSampler] groups:
  key=(K,D,H,W)=(3, 20, 256, 256), n=51
  key=(K,D,H,W)=(3, 30, 256, 256), n=21
  key=(K,D,H,W)=(5, 20, 256, 256), n=60
  key=(K,D,H,W)=(5, 30, 256, 256), n=12
  key=(K,D,H,W)=(7, 20, 256, 256), n=62
  key=(K,D,H,W)=(7, 30, 256, 256), n=10
2026-05-09 13:56:19 | [Logger] log file: ./checkpoints/coarseflow_v6_stage2_varK_varSpacing/train.log
2026-05-09 13:56:19 | [Resume] Loading checkpoint: ./checkpoints/coarseflow_v6_stage2_varK_varSpacing/latest.pth
2026-05-09 13:56:19 | [Resume] start_epoch = 348


/home/cyf/wbi/Virginia/code/CoarseFlow/training/train.py:568: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=use_amp)
/home/cyf/wbi/Virginia/code/CoarseFlow/training/train.py:155: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):


2026-05-09 13:56:22 | [Epoch 348] step 0000/379 | loss=0.8854 | match=0.7674 | kl=0.2645 | ce=1.6763 | coord=1.0832 | disp=1.0832 | smooth=1.2580 | z_spacing=0.6648 | top1=0.762 | top5=0.987 | pmax=0.225 | ent=3.355 | inside_valid=1.000 | cand_valid=0.813 | inside_all=0.909 | valid_inside_all=0.624 | time=2.2s | valid=0.624 | 
2026-05-09 13:56:33 | [Epoch 348] step 0010/379 | loss=0.9669 | match=0.8510 | kl=0.3660 | ce=1.6166 | coord=1.0395 | disp=1.0395 | smooth=1.3988 | z_spacing=0.9919 | top1=0.785 | top5=0.982 | pmax=0.237 | ent=3.328 | inside_valid=1.000 | cand_valid=0.769 | inside_all=0.904 | valid_inside_all=0.635 | time=13.3s | valid=0.635 | 
2026-05-09 13:56:42 | [Epoch 348] step 0020/379 | loss=1.0991 | match=0.9401 | kl=0.3703 | ce=1.8994 | coord=1.2768 | disp=1.2768 | smooth=1.4626 | z_spacing=4.7991 | top1=0.635 | top5=0.949 | pmax=0.207 | ent=3.394 | inside_valid=1.000 | cand_valid=0.790 | inside_all=0.885 | valid_inside_all=0.597 | time=23.1s | valid=0.597 | 
2026-05-09 

/home/cyf/wbi/Virginia/code/CoarseFlow/training/train.py:317: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):


2026-05-09 14:03:48 | [Val Epoch 348] loss=1.0823 | match=0.9397 | kl=0.3881 | ce=1.8388 | coord=1.2659 | disp=1.2659 | smooth=1.5273 | z_spacing=1.6735 | top1=0.675 | top5=0.956 | pmax=0.213 | ent=3.398 | inside_valid=1.000 | cand_valid=0.813 | inside_all=0.864 | valid_inside_all=0.608 | valid=0.608 | pred_mag=2.419 | gt_mag=2.582
2026-05-09 14:03:49 | [Epoch 349] step 0000/379 | loss=1.1160 | match=0.9748 | kl=0.3933 | ce=1.9380 | coord=1.2486 | disp=1.2486 | smooth=1.6251 | z_spacing=1.6577 | top1=0.641 | top5=0.937 | pmax=0.196 | ent=3.393 | inside_valid=1.000 | cand_valid=0.750 | inside_all=0.804 | valid_inside_all=0.549 | time=1.2s | valid=0.549 | 
2026-05-09 14:04:01 | [Epoch 349] step 0010/379 | loss=1.1269 | match=0.9327 | kl=0.3605 | ce=1.9073 | coord=1.7132 | disp=1.7132 | smooth=1.5481 | z_spacing=3.0180 | top1=0.631 | top5=0.951 | pmax=0.209 | ent=3.394 | inside_valid=1.000 | cand_valid=0.779 | inside_all=0.839 | valid_inside_all=0.640 | time=12.4s | valid=0.640 | 
2026-05

Stage 2.5, add the attention
---

In [ ]:
train_loader_attn_warmup, _, _ = build_sameShape_loader(
    manifest_summary["stage2_train"],
    batch_size=2,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
    drop_last=False,
)

val_loader_attn_warmup, _, _ = build_sameShape_loader(
    manifest_summary["stage2_val"],
    batch_size=1,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
    drop_last=False,
)
model_config_attn.update(
    dict(
        # moving attention
        moving_window_attn_layers=2,
        num_refine_iters=3,
        # reference attention
        ref_use_attention=True,
        ref_attn_layers=2,
        ref_attn_num_heads=4,
        ref_attn_window_size=(3, 8, 8),
        ref_attn_mlp_ratio=2.0,
    )

)
model_stage2_5_attn = train_coarse_matching_model(
    train_dataset=None,
    val_dataset=None,
    train_loader=train_loader_attn_warmup,
    val_loader=val_loader_attn_warmup,

    save_dir="./checkpoints/coarseflow_v6_stage2_5_attn_warmup_iter=3",

    num_epochs=100,
    lr=1e-5,
    batch_size=2,
    num_workers=0,
    use_amp=True,

    **model_config_attn,

    # 不要一开始 sharpen
    loss_mode="match",
    lambda_match=1.0,
    lambda_match_kl=0.0,
    lambda_match_ce=0.5,
    lambda_coord=0.2,
    lambda_disp=0.0,
    lambda_smooth=0.008,
    lambda_z_spacing=0.005,

    # 从无 attention 的 Stage 2 checkpoint 继续
    resume_path="./checkpoints/coarseflow_v6_stage2_varK_varSpacing/best.pth",
    resume_optimizer=False,
    strict_load=False,

    log_filename="train.log",
    log_mode="w",
    freeze_encoders=False
)

[SameShapeBatchSampler] groups:
  key=(K,D,H,W)=(3, 20, 256, 256), n=230
  key=(K,D,H,W)=(3, 30, 256, 256), n=70
  key=(K,D,H,W)=(4, 20, 256, 256), n=235
  key=(K,D,H,W)=(4, 30, 256, 256), n=65
  key=(K,D,H,W)=(5, 20, 256, 256), n=217
  key=(K,D,H,W)=(5, 30, 256, 256), n=83
  key=(K,D,H,W)=(6, 20, 256, 256), n=223
  key=(K,D,H,W)=(6, 30, 256, 256), n=77
  key=(K,D,H,W)=(7, 20, 256, 256), n=224
  key=(K,D,H,W)=(7, 30, 256, 256), n=76
[SameShapeBatchSampler] groups:
  key=(K,D,H,W)=(3, 20, 256, 256), n=51
  key=(K,D,H,W)=(3, 30, 256, 256), n=21
  key=(K,D,H,W)=(5, 20, 256, 256), n=60
  key=(K,D,H,W)=(5, 30, 256, 256), n=12
  key=(K,D,H,W)=(7, 20, 256, 256), n=62
  key=(K,D,H,W)=(7, 30, 256, 256), n=10
2026-05-12 20:17:57 | [Logger] log file: ./checkpoints/coarseflow_v6_stage2_5_attn_warmup_iter=3/train.log
2026-05-12 20:17:57 | [Resume] Loading checkpoint: ./checkpoints/coarseflow_v6_stage2_varK_varSpacing/best.pth
2026-05-12 20:17:57 | [Resume] load_state_dict message: _IncompatibleKeys

/home/cyf/wbi/Virginia/code/CoarseFlow/training/train.py:630: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=use_amp)
/home/cyf/wbi/Virginia/code/CoarseFlow/training/train.py:160: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):


2026-05-12 20:17:59 | [Epoch 407] step 0000/753 | loss=2.3488 | match=1.5665 | kl=1.4856 | ce=3.1330 | coord=3.6570 | disp=3.6570 | smooth=5.1778 | z_spacing=1.8940 | top1=0.447 | top5=0.676 | pmax=0.162 | ent=3.925 | inside_valid=1.000 | cand_valid=0.760 | inside_all=0.993 | valid_inside_all=0.657 | time=1.8s | valid=0.659 | 
2026-05-12 20:18:09 | [Epoch 407] step 0010/753 | loss=1.6499 | match=1.1421 | kl=0.9230 | ce=2.2841 | coord=2.3755 | disp=2.3755 | smooth=3.5264 | z_spacing=0.9129 | top1=0.625 | top5=0.825 | pmax=0.232 | ent=3.565 | inside_valid=1.000 | cand_valid=0.816 | inside_all=0.998 | valid_inside_all=0.622 | time=11.6s | valid=0.622 | 
2026-05-12 20:18:20 | [Epoch 407] step 0020/753 | loss=1.0785 | match=0.7347 | kl=0.7219 | ce=1.4693 | coord=1.5518 | disp=1.5518 | smooth=2.1974 | z_spacing=3.1759 | top1=0.761 | top5=0.978 | pmax=0.323 | ent=2.985 | inside_valid=1.000 | cand_valid=0.785 | inside_all=1.000 | valid_inside_all=0.599 | time=23.1s | valid=0.599 | 


TEST ITERATION
---

In [7]:
import torch
import numpy as np
import pandas as pd

from CoarseFlow.models.SparseGMFlow3D import CoarseMatchingNetV5
from CoarseFlow.training import train
def set_refine_iters(model, n):
    """
    Dynamically set refinement iterations.
    """
    n = int(n)
    changed = False

    if hasattr(model, "num_refine_iters"):
        model.num_refine_iters = n
        changed = True

    for m in model.modules():
        if m is not model and hasattr(m, "num_refine_iters"):
            m.num_refine_iters = n
            changed = True

    if not changed:
        raise AttributeError("No num_refine_iters found in model.")


def set_query_chunk_size(model, chunk_size):
    """
    Reduce query chunk size if inference OOM.
    """
    for m in model.modules():
        if hasattr(m, "query_chunk_size"):
            m.query_chunk_size = int(chunk_size)


def metric_to_float(v):
    if torch.is_tensor(v):
        return float(v.detach().float().mean().cpu())
    if isinstance(v, np.ndarray):
        return float(np.mean(v))
    return float(v)

val_loader_attn_warmup, _, _ = build_sameShape_loader(
    manifest_summary["stage2_val"],
    batch_size=1,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
    drop_last=False,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ckpt_path = "/home/cyf/wbi/Virginia/code/CoarseFlow/scripys/checkpoints/coarseflow_v6_stage2_5_attn_warmup/best.pth"

model_config_eval = dict(model_config_attn)
model_config_eval["num_refine_iters"] = 10

model = CoarseMatchingNetV5(**model_config_eval).to(device)

ckpt = torch.load(
    ckpt_path,
    map_location=device,
    weights_only=False,
)

model.load_state_dict(ckpt["model"], strict=True)
model.eval()

print("Loaded:", ckpt_path)
print("epoch:", ckpt.get("epoch", "unknown"))
print("best_val_loss:", ckpt.get("best_val_loss", "unknown"))
# 取一个 batch
sample_batch = next(iter(val_loader_attn_warmup))

model.eval()
iteration=10
with torch.no_grad():
    set_refine_iters(model, 1)
    out1 = train.inference(
        model=model,
        sample_or_batch=sample_batch,
        device=device,
        build_valid_mask=True,
    )

    set_refine_iters(model, iteration)
    out10 = train.inference(
        model=model,
        sample_or_batch=sample_batch,
        device=device,
        build_valid_mask=True,
    )

pred1 = out1["pred_coords"].detach().cpu()
pred10 = out10["pred_coords"].detach().cpu()

diff = (pred1 - pred10).abs()

print("===== iter = 1 metrics =====")
print(out1["metrics"])

print(f"\n===== iter = {iteration} metrics =====")
print(out10["metrics"])

print("\n===== pred_coords difference =====")
print("max diff :", diff.max().item())
print("mean diff:", diff.mean().item())

[SameShapeBatchSampler] groups:
  key=(K,D,H,W)=(3, 20, 256, 256), n=51
  key=(K,D,H,W)=(3, 30, 256, 256), n=21
  key=(K,D,H,W)=(5, 20, 256, 256), n=60
  key=(K,D,H,W)=(5, 30, 256, 256), n=12
  key=(K,D,H,W)=(7, 20, 256, 256), n=62
  key=(K,D,H,W)=(7, 30, 256, 256), n=10
Loaded: /home/cyf/wbi/Virginia/code/CoarseFlow/scripys/checkpoints/coarseflow_v6_stage2_5_attn_warmup/best.pth
epoch: 423
best_val_loss: 1.3240524715295545
scores shape: torch.Size([1, 768, 1089])
candidate M: 1089
scores shape: torch.Size([1, 768, 1089])
candidate M: 1089
===== iter = 1 metrics =====
{'disp_mae_all': tensor(0.4355), 'disp_mae_valid': tensor(0.4740), 'z_disp_mae_all': tensor(0.3556), 'z_disp_mae_valid': tensor(0.3791), 'coord_mae_all': tensor(0.4355), 'coord_mae_valid': tensor(0.4740), 'z_coord_mae_all': tensor(0.3556), 'z_coord_mae_valid': tensor(0.3791), 'valid_ratio': tensor(0.5612), 'prob_max': tensor(0.3384), 'entropy': tensor(2.9596)}

===== iter = 10 metrics =====
{'disp_mae_all': tensor(0.7005)

Stage3，  concrete coord prediction
---

In [4]:
# ============================================================
# Stage 3a: multi-iteration sharpen, iter = 3
# ============================================================

train_loader_stage3a, _, _ = build_sameShape_loader(
    manifest_summary["stage3_train"],
    batch_size=2,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
    drop_last=False,
)

val_loader_stage3a, _, _ = build_sameShape_loader(
    manifest_summary["stage3_val"],
    batch_size=1,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
    drop_last=False,
)

model_config_stage3a = dict(model_config)

model_config_stage3a.update(
    dict(
        # ----------------------------------------------------
        # keep architecture identical to the checkpoint
        # ----------------------------------------------------
        moving_window_attn_layers=2,

        ref_use_attention=True,
        ref_attn_layers=2,
        ref_attn_num_heads=4,
        ref_attn_window_size=(3, 8, 8),
        ref_attn_mlp_ratio=2.0,

        # ----------------------------------------------------
        # key change for stage 3
        # ----------------------------------------------------
        num_refine_iters=3,
    )
)

# model_stage3a = train_coarse_matching_model(
#     train_dataset=None,
#     val_dataset=None,
#     train_loader=train_loader_stage3a,
#     val_loader=val_loader_stage3a,

#     save_dir="./checkpoints/coarseflow_v6_stage3a_iter3_sharpen",

#     num_epochs=100,
#     lr=5e-6,
#     batch_size=2,
#     num_workers=0,
#     use_amp=True,

#     **model_config_stage3a,

#     # --------------------------------------------------------
#     # loss
#     # --------------------------------------------------------
#     loss_mode="match",
#     lambda_match=1.0,
#     lambda_match_kl=0.4,
#     lambda_match_ce=0.5,
#     lambda_coord=0.3,
#     lambda_disp=0.0,
#     lambda_smooth=0.005,
#     lambda_z_spacing=0.003,

#     # --------------------------------------------------------
#     # important: use chunked CE/KL, not full scores output
#     # --------------------------------------------------------
#     compute_chunk_match_loss=True,
#     match_sigma=(0.5, 1.0, 1.0),
#     match_inside_threshold=4.0,

#     # --------------------------------------------------------
#     # resume
#     # --------------------------------------------------------
#     resume_path="/home/cyf/wbi/Virginia/code/CoarseFlow/scripys/checkpoints/coarseflow_v6_stage2_5_attn_warmup/best.pth",
#     resume_optimizer=False,
#     strict_load=True,

#     log_filename="train.log",
#     log_mode="a",
# )
model_stage3a = train_coarse_matching_model(
    train_dataset=None,
    val_dataset=None,
    train_loader=train_loader_stage3a,
    val_loader=val_loader_stage3a,

    save_dir="./checkpoints/coarseflow_v6_stage3a_iter3_sharpen",

    num_epochs=30,
    lr=5e-6,
    batch_size=2,
    num_workers=0,
    use_amp=True,

    **model_config_stage3a,

    # --------------------------------------------------------
    # loss
    # --------------------------------------------------------
    loss_mode="match",
    lambda_match=1.0,
    lambda_match_kl=0.4,
    lambda_match_ce=0.5,
    lambda_coord=0.3,
    lambda_disp=0.0,
    lambda_smooth=0.005,
    lambda_z_spacing=0.003,

    # --------------------------------------------------------
    # important: use chunked CE/KL, not full scores output
    # --------------------------------------------------------
    compute_chunk_match_loss=True,
    match_sigma=(0.5, 1.0, 1.0),
    match_inside_threshold=4.0,

    # --------------------------------------------------------
    # resume
    # --------------------------------------------------------
    resume_path="/home/cyf/wbi/Virginia/code/CoarseFlow/scripys/checkpoints/coarseflow_v6_stage3a_iter3_sharpen/latest.pth",
    resume_optimizer=False,
    strict_load=True,

    log_filename="train.log",
    log_mode="a",
)

[SameShapeBatchSampler] groups:
  key=(K,D,H,W)=(3, 20, 256, 256), n=110
  key=(K,D,H,W)=(3, 30, 256, 256), n=25
  key=(K,D,H,W)=(5, 20, 256, 256), n=108
  key=(K,D,H,W)=(5, 30, 256, 256), n=27
  key=(K,D,H,W)=(7, 20, 256, 256), n=99
  key=(K,D,H,W)=(7, 30, 256, 256), n=36
[SameShapeBatchSampler] groups:
  key=(K,D,H,W)=(3, 20, 256, 256), n=39
  key=(K,D,H,W)=(3, 30, 256, 256), n=9
  key=(K,D,H,W)=(5, 20, 256, 256), n=38
  key=(K,D,H,W)=(5, 30, 256, 256), n=10
  key=(K,D,H,W)=(7, 20, 256, 256), n=37
  key=(K,D,H,W)=(7, 30, 256, 256), n=11
2026-05-13 09:38:00 | [Logger] log file: ./checkpoints/coarseflow_v6_stage3a_iter3_sharpen/train.log
2026-05-13 09:38:00 | [Resume] Loading checkpoint: /home/cyf/wbi/Virginia/code/CoarseFlow/scripys/checkpoints/coarseflow_v6_stage3a_iter3_sharpen/latest.pth
2026-05-13 09:38:00 | [Resume] start_epoch = 532


/home/cyf/wbi/Virginia/code/CoarseFlow/training/train.py:630: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=use_amp)
/home/cyf/wbi/Virginia/code/CoarseFlow/training/train.py:160: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):


2026-05-13 09:38:02 | [Epoch 532] step 0000/204 | loss=1.1202 | match=0.7707 | kl=0.7341 | ce=0.9541 | coord=1.1212 | disp=1.1212 | smooth=2.2962 | z_spacing=0.5761 | top1=0.845 | top5=0.982 | pmax=0.513 | ent=2.382 | inside_valid=1.000 | cand_valid=0.827 | inside_all=0.978 | valid_inside_all=0.618 | time=1.8s | valid=0.618 | 
2026-05-13 09:38:12 | [Epoch 532] step 0010/204 | loss=1.6575 | match=1.0160 | kl=0.7788 | ce=1.4091 | coord=2.0485 | disp=2.0485 | smooth=2.6628 | z_spacing=4.5219 | top1=0.725 | top5=0.900 | pmax=0.459 | ent=2.560 | inside_valid=1.000 | cand_valid=0.729 | inside_all=0.979 | valid_inside_all=0.674 | time=12.3s | valid=0.674 | 
2026-05-13 09:38:24 | [Epoch 532] step 0020/204 | loss=1.3401 | match=0.8631 | kl=0.7342 | ce=1.1389 | coord=1.5290 | disp=1.5290 | smooth=3.1598 | z_spacing=0.8008 | top1=0.779 | top5=0.956 | pmax=0.487 | ent=2.460 | inside_valid=1.000 | cand_valid=0.754 | inside_all=0.972 | valid_inside_all=0.535 | time=23.6s | valid=0.535 | 
2026-05-13 

/home/cyf/wbi/Virginia/code/CoarseFlow/training/train.py:339: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):


2026-05-13 09:42:07 | [Val Epoch 532] loss=1.4688 | match=0.9828 | kl=0.8722 | ce=1.2678 | coord=1.5500 | disp=1.5500 | smooth=2.7170 | z_spacing=2.4868 | top1=0.751 | top5=0.939 | pmax=0.485 | ent=2.463 | inside_valid=1.000 | cand_valid=0.781 | inside_all=0.980 | valid_inside_all=0.600 | valid=0.600 | pred_mag=4.342 | gt_mag=4.456
2026-05-13 09:42:09 | [Epoch 533] step 0000/204 | loss=1.1255 | match=0.7652 | kl=0.7378 | ce=0.9402 | coord=1.1355 | disp=1.1355 | smooth=2.0366 | z_spacing=3.1468 | top1=0.853 | top5=0.979 | pmax=0.517 | ent=2.357 | inside_valid=1.000 | cand_valid=0.763 | inside_all=0.978 | valid_inside_all=0.508 | time=1.3s | valid=0.508 | 
2026-05-13 09:42:19 | [Epoch 533] step 0010/204 | loss=1.5484 | match=1.0415 | kl=0.8704 | ce=1.3867 | coord=1.6152 | disp=1.6152 | smooth=2.3940 | z_spacing=3.4491 | top1=0.723 | top5=0.911 | pmax=0.457 | ent=2.575 | inside_valid=1.000 | cand_valid=0.834 | inside_all=0.976 | valid_inside_all=0.577 | time=11.8s | valid=0.577 | 
2026-05

In [ ]:
# ============================================================
# Stage 3b: multi-iteration sharpen, iter = 3
# ============================================================

train_loader_stage3b, _, _ = build_sameShape_loader(
    manifest_summary["stage3_train"],
    batch_size=2,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
    drop_last=False,
)

val_loader_stage3b, _, _ = build_sameShape_loader(
    manifest_summary["stage3_val"],
    batch_size=1,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
    drop_last=False,
)

model_config_stage3b = dict(model_config)

model_config_stage3b.update(
    dict(
        # ----------------------------------------------------
        # keep architecture identical to the checkpoint
        # ----------------------------------------------------
        moving_window_attn_layers=2,

        ref_use_attention=True,
        ref_attn_layers=2,
        ref_attn_num_heads=4,
        ref_attn_window_size=(3, 8, 8),
        ref_attn_mlp_ratio=2.0,

        # ----------------------------------------------------
        # key change for stage 3
        # ----------------------------------------------------
        num_refine_iters=3,
    )
)

model_stage3b = train_coarse_matching_model(
    train_dataset=None,
    val_dataset=None,
    train_loader=train_loader_stage3b,
    val_loader=val_loader_stage3b,

    save_dir="./checkpoints/coarseflow_v6_stage3b_iter3_sharpen",

    num_epochs=100,
    lr=5e-6,
    batch_size=2,
    num_workers=0,
    use_amp=True,

    **model_config_stage3b,

    # --------------------------------------------------------
    # loss
    # --------------------------------------------------------
    loss_mode="match",
    lambda_match=1.0,
    lambda_match_kl=0.25,
    lambda_match_ce=0.45,
    lambda_coord=0.6,
    lambda_disp=0.0,
    lambda_smooth=0.005,
    lambda_z_spacing=0.003,

    # --------------------------------------------------------
    # important: use chunked CE/KL, not full scores output
    # --------------------------------------------------------
    compute_chunk_match_loss=True,
    match_sigma=(0.5, 1.0, 1.0),
    match_inside_threshold=4.0,

    # --------------------------------------------------------
    # resume
    # --------------------------------------------------------
    resume_path="/home/cyf/wbi/Virginia/code/CoarseFlow/scripys/checkpoints/coarseflow_v6_stage3a_iter3_sharpen/best.pth",
    resume_optimizer=False,
    strict_load=True,

    log_filename="train.log",
    log_mode="a",
    
)

Stage 4 fine training
---


In [4]:
train_loader_stage4a, _, _ = build_sameShape_loader(
    manifest_summary["stage3_train"],
    batch_size=32,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
    drop_last=False,
)

val_loader_stage4a, _, _ = build_sameShape_loader(
    manifest_summary["stage3_val"],
    batch_size=1,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
    drop_last=False,
)

model_config_residual = dict(model_config_attn)

model_config_residual.update(
    dict(
        num_refine_iters=3,

        use_coord_residual=True,
        residual_type="spatial",

        residual_hidden_dim=256,
        residual_num_blocks=5,
        residual_max_delta=(1.5, 3.0, 3.0),
        residual_use_disp=True,
        residual_use_3d=True,

        residual_detach_coarse=True,
        residual_detach_features=True,
    )
)
model_stage4_spatial = train_coarse_matching_model(
    train_dataset=None,
    val_dataset=None,
    train_loader=train_loader_stage4a,
    val_loader=val_loader_stage4a,

    save_dir="./checkpoints/coarseflow_v6_stage4_spatial_residual",

    num_epochs=150,
    lr=3e-4,
    residual_lr=3e-4,
    weight_decay=0.0,
    batch_size=32,
    num_workers=0,
    use_amp=True,

    **model_config_residual,

    loss_mode="coord",
    lambda_match=0.0,
    lambda_match_kl=0.0,
    lambda_match_ce=0.0,
    lambda_coord=1.0,
    lambda_disp=0.0,
    lambda_smooth=0.0,
    lambda_z_spacing=0.0,

    compute_chunk_match_loss=False,

    train_only_residual=True,
    freeze_encoder=False,

    resume_path="/home/cyf/wbi/Virginia/code/CoarseFlow/scripys/checkpoints/coarseflow_v6_stage3b_iter3_sharpen/best.pth",
    resume_optimizer=False,
    strict_load=False,

    log_filename="train.log",
    log_mode="w",
)

[SameShapeBatchSampler] groups:
  key=(K,D,H,W)=(3, 20, 256, 256), n=110
  key=(K,D,H,W)=(3, 30, 256, 256), n=25
  key=(K,D,H,W)=(5, 20, 256, 256), n=108
  key=(K,D,H,W)=(5, 30, 256, 256), n=27
  key=(K,D,H,W)=(7, 20, 256, 256), n=99
  key=(K,D,H,W)=(7, 30, 256, 256), n=36
[SameShapeBatchSampler] groups:
  key=(K,D,H,W)=(3, 20, 256, 256), n=39
  key=(K,D,H,W)=(3, 30, 256, 256), n=9
  key=(K,D,H,W)=(5, 20, 256, 256), n=38
  key=(K,D,H,W)=(5, 30, 256, 256), n=10
  key=(K,D,H,W)=(7, 20, 256, 256), n=37
  key=(K,D,H,W)=(7, 30, 256, 256), n=11
2026-05-14 11:30:16 | [Logger] log file: ./checkpoints/coarseflow_v6_stage4_spatial_residual/train.log
2026-05-14 11:30:16 | [Resume] Loading checkpoint: /home/cyf/wbi/Virginia/code/CoarseFlow/scripys/checkpoints/coarseflow_v6_stage3b_iter3_sharpen/best.pth
2026-05-14 11:30:16 | [Resume] load_state_dict message: _IncompatibleKeys(missing_keys=['coord_residual_refiner.in_proj.0.weight', 'coord_residual_refiner.in_proj.0.bias', 'coord_residual_refiner.i

/home/cyf/wbi/Virginia/code/CoarseFlow/training/train.py:785: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=use_amp)
/home/cyf/wbi/Virginia/code/CoarseFlow/training/train.py:160: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):


2026-05-14 11:30:22 | [Epoch 577] step 0000/16 | loss=1.2277 | match=0.0000 | kl=0.0000 | ce=0.0000 | coord=1.2277 | disp=1.2277 | smooth=2.8840 | z_spacing=0.6766 | top1=0.000 | top5=0.000 | pmax=0.000 | ent=0.000 | inside_valid=0.000 | cand_valid=0.000 | inside_all=0.000 | valid_inside_all=0.000 | time=5.1s | valid=0.597 | 
2026-05-14 11:31:08 | [Epoch 577] step 0010/16 | loss=1.3277 | match=-0.0000 | kl=-0.0000 | ce=-0.0000 | coord=1.3277 | disp=1.3277 | smooth=2.5542 | z_spacing=1.9507 | top1=0.000 | top5=0.000 | pmax=0.000 | ent=0.000 | inside_valid=0.000 | cand_valid=0.000 | inside_all=0.000 | valid_inside_all=0.000 | time=50.9s | valid=0.648 | 
2026-05-14 11:31:25 | [Epoch 577] train_loss=1.3054


/home/cyf/wbi/Virginia/code/CoarseFlow/training/train.py:339: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):


2026-05-14 11:31:53 | [Val Epoch 577] loss=1.5016 | match=0.0000 | kl=0.0000 | ce=0.0000 | coord=1.5016 | disp=1.5016 | smooth=2.7217 | z_spacing=2.4804 | top1=0.000 | top5=0.000 | pmax=0.000 | ent=0.000 | inside_valid=0.000 | cand_valid=0.000 | inside_all=0.000 | valid_inside_all=0.000 | valid=0.600 | pred_mag=4.392 | gt_mag=4.456
2026-05-14 11:31:58 | [Epoch 578] step 0000/16 | loss=1.2237 | match=0.0000 | kl=0.0000 | ce=0.0000 | coord=1.2237 | disp=1.2237 | smooth=2.4527 | z_spacing=2.7069 | top1=0.000 | top5=0.000 | pmax=0.000 | ent=0.000 | inside_valid=0.000 | cand_valid=0.000 | inside_all=0.000 | valid_inside_all=0.000 | time=5.1s | valid=0.550 | 
2026-05-14 11:32:37 | [Epoch 578] step 0010/16 | loss=1.1727 | match=0.0000 | kl=0.0000 | ce=0.0000 | coord=1.1727 | disp=1.1727 | smooth=2.3002 | z_spacing=2.6768 | top1=0.000 | top5=0.000 | pmax=0.000 | ent=0.000 | inside_valid=0.000 | cand_valid=0.000 | inside_all=0.000 | valid_inside_all=0.000 | time=43.9s | valid=0.558 | 
2026-05-1

KeyboardInterrupt: 

Stage 4 hard fine-tune
---

In [ ]:
# Stage 4: hard fine-tune, optional

train_loader_stage3_hard, _, _ = build_sameShape_loader(
    manifest_summary["stage3_train"],
    batch_size=4,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
    drop_last=False,
)

val_loader_stage3_hard, _, _ = build_sameShape_loader(
    manifest_summary["stage3_val"],
    batch_size=1,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
    drop_last=False,
)

model_stage4 = train_coarse_matching_model(
    train_dataset=None,
    val_dataset=None,
    train_loader=train_loader_stage3_hard,
    val_loader=val_loader_stage3_hard,

    save_dir="./checkpoints/coarseflow_v6_stage4_hard_finetune",

    num_epochs=30,
    lr=5e-6,
    batch_size=4,
    num_workers=0,
    use_amp=True,

    **model_config,

    loss_mode="match",
    lambda_match=1.0,
    lambda_match_kl=0.7,
    lambda_match_ce=0.5,
    lambda_coord=0.2,
    lambda_disp=0.0,
    lambda_smooth=0.01,
    lambda_z_spacing=0.005,

    resume_path="./checkpoints/coarseflow_v6_stage3_varK_sharpen/best.pth",
    resume_optimizer=False,
    strict_load=True,

    log_filename="train.log",
    log_mode="w",
)

Stage 5 Eventual Evaluation
---

In [ ]:
# train.ipynb - Cell 10
# Final test evaluation

from CoarseFlow.models.SparseGMFlow3D import CoarseMatchingNetV5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

test_loader, _, _ = build_sameShape_loader(
    manifest_summary["final_test"],
    batch_size=1,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
    drop_last=False,
)

model = CoarseMatchingNetV5(**model_config).to(device)

ckpt_path = "./checkpoints/coarseflow_v6_stage3_varK_sharpen/best.pth"
ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)

load_msg = model.load_state_dict(ckpt["model"], strict=True)
print("loaded:", ckpt_path)
print("epoch:", ckpt.get("epoch"))
print("best_val_loss:", ckpt.get("best_val_loss"))
print(load_msg)

test_loss = train.validate_one_epoch(
    model=model,
    loader=test_loader,
    device=device,
    epoch=0,
    use_amp=True,

    loss_mode="match",
    lambda_match=1.0,
    lambda_match_kl=0.7,
    lambda_match_ce=0.6,
    lambda_coord=0.15,
    lambda_disp=0.0,
    lambda_smooth=0.005,
    lambda_z_spacing=0.005,
)

print("test_loss:", test_loss)
# train.ipynb - Cell 11



In [19]:
from importlib import reload
from CoarseFlow.training import diagnostics
from CoarseFlow.models.SparseGMFlow3D import CoarseMatchingNetV5
reload(diagnostics)
model_config =dict(
    dim=96,
    radius=(4, 5, 5),
    temperature=0.05,
    use_learned_matching=True,
    matcher_mode="hybrid",

    control_stride=16,
    encoder_stride=4,
    num_refine_iters=1,
    query_chunk_size=128,

    moving_num_convnext_blocks=3,
    moving_window_attn_layers=2,

    # reference attention
    ref_use_attention=True,
    ref_attn_layers=2,
    ref_attn_num_heads=4,
    ref_attn_window_size=(3, 8, 8),
    ref_attn_mlp_ratio=2.0,
    ref_base_channels=(32, 64, 96),
    ref_num_blocks=(2, 4, 4),
    ref_refine_blocks=3,
    use_coord_embed=True,

    use_offset_encoding=True,
    use_offset_bias=True,
    use_local_cross_attn=True,
    local_attn_temperature=0.20,

    use_spacing_embed=True,
)
model_config_attn = dict(model_config)

model_config_attn.update(
    dict(
        # moving attention
        moving_window_attn_layers=2,

        # reference attention
        ref_use_attention=True,
        ref_attn_layers=2,
        ref_attn_num_heads=4,
        ref_attn_window_size=(3, 8, 8),
        ref_attn_mlp_ratio=2.0,
        num_refine_iters=3,
    )

)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CoarseMatchingNetV5(**model_config_attn).to(device)
val_loader_stage3_hard, _, _ = build_sameShape_loader(
    manifest_summary["stage3_val"],
    batch_size=1,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
    drop_last=False,
)

ckpt_path = "/home/cyf/wbi/Virginia/code/CoarseFlow/scripys/checkpoints/coarseflow_v6_stage3b_iter3_sharpen/best.pth"
ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)

load_msg = model.load_state_dict(ckpt["model"], strict=True)
diag_df = diagnostics.run_coord_diagnosis(
    model=model,   # 或者你自己 load 的 model
    loader=val_loader_stage3_hard,
    device = device,
    max_batches=50,
    use_amp=True,
    sigma=(0.5, 1.0, 1.0),
    inside_threshold=4.0,
)

diag_df.to_csv("/home/cyf/wbi/Virginia/code/CoarseFlow/scripys/checkpoints/coarseflow_v6_stage3b_iter3_sharpen/coord_diagnosis.csv", index=False)

[SameShapeBatchSampler] groups:
  key=(K,D,H,W)=(3, 20, 256, 256), n=39
  key=(K,D,H,W)=(3, 30, 256, 256), n=9
  key=(K,D,H,W)=(5, 20, 256, 256), n=38
  key=(K,D,H,W)=(5, 30, 256, 256), n=10
  key=(K,D,H,W)=(7, 20, 256, 256), n=37
  key=(K,D,H,W)=(7, 30, 256, 256), n=11

===== Diagnosis mean over batches =====
num_eval_points                    : 464.340000
valid_ratio_all                    : 0.587839
inside_ratio_valid                 : 0.999642
top1                               : 0.782452
top5                               : 0.957750
pmax_mean                          : 0.534452
entropy_mean                       : 2.229547
pmax_top1_correct                  : 0.559248
pmax_top1_wrong                    : 0.442791
entropy_top1_correct               : 2.159765
entropy_top1_wrong                 : 2.485587
soft_l1_all                        : 1.397742
soft_l1_top1_correct               : 1.097806
soft_l1_topk_not_top1              : 2.071723
soft_l1_topk_wrong                 : 3.929